# 2D Allen-Cahn Equation
## Phase-Field Dynamics, Mean Curvature Flow, and Domain Coarsening

This notebook simulates the 2D Allen-Cahn equation using the pseudo-spectral `PDESolver` framework. The Allen-Cahn equation is a foundational model in phase-field theory, describing the motion of phase boundaries (interfaces) in multi-phase materials. It is the gradient flow of the Ginzburg-Landau free energy functional and drives the system toward a state that minimizes total interfacial area (mean curvature flow).

---

## 1. The Governing Equation

$$
\partial_t u = \epsilon^2 \nabla^2 u + u - u^3
$$

* $u(x,y,t)$ (Order Parameter): Represents the local phase of the material. The stable bulk phases correspond to $u = +1$ and $u = -1$, while $u = 0$ represents the diffuse interface between them.
* $\epsilon$ (Interface Width Parameter): Controls the thickness of the diffuse interface. Smaller $\epsilon$ leads to sharper, more distinct boundaries.
* $u - u^3$ (Nonlinear Reaction): Derived from the derivative of a double-well potential $F(u) = \frac{1}{4}(1-u^2)^2$. It drives the system away from the unstable $u=0$ state and pulls it toward the stable $u=\pm 1$ states.
* $\epsilon^2 \nabla^2 u$ (Diffusion/Interfacial Energy): Penalizes sharp gradients, smoothing out the interface and providing the surface tension that drives curvature-driven motion.

---

## 2. Reformulation for the Solver

We separate the equation into a linear diffusive part (handled in Fourier space) and a nonlinear reaction part (handled in physical space).

In Fourier space, $\nabla^2 \to -(\xi^2 + \eta^2)$, so the linear operator becomes:

$$
\text{Linear symbol:} \quad -\epsilon^2(\xi^2 + \eta^2)
$$

The equation in the solver's format:

$$
\partial_t u = \underbrace{u_{\text{op}}\!\left(-\epsilon^2(\xi^2 + \eta^2)\right)u} {\text{Linear diffusion (Fourier space)}} + \underbrace{u - u^3} {\text{Double-well reaction (Physical space)}}
$$

---

## 3. Physical Phenomena

* **Spinodal Decomposition**: Starting from a homogeneous unstable state (small random noise around $u=0$), the system rapidly phase-separates into a labyrinthine mixture of $u \approx +1$ and $u \approx -1$ domains.
* **Mean Curvature Flow**: Once the sharp interfaces form, they begin to move. Interfaces curve to minimize their length, causing small domains to shrink and disappear while larger domains grow.
* **Domain Coarsening**: Over long times, the characteristic length scale of the pattern grows algebraically ($L(t) \propto \sqrt{t}$), leading to fewer, larger, and smoother domains.

We initialize the system with **small random noise** to trigger the instability and watch the beautiful curvature-driven coarsening unfold.

# Implementation
## 0. Imports 

In [ ]:
from solver import PDESolver, psiOp  # psiOp for real-valued fields
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML

## 1. Physical and simulation parameters 

In [ ]:
# ── Allen-Cahn Coefficients ──
EPSILON = 0.05    # Interface width (smaller = sharper interfaces)

# ── Grid and Time ──
# Domain size and resolution must resolve the interface width (epsilon)
Lx, Ly = 2.0, 2.0
Nx, Ny = 256, 256    

# Coarsening is a slow process; we need a long integration time
Lt, Nt = 2.0, 2000
n_frames = 200

## 2. Grid setup 

In [ ]:
xs_1d = np.linspace(-Lx/2, Lx/2, Nx)
ys_1d = np.linspace(-Ly/2, Ly/2, Ny)

# CRITICAL FIX: psipy internally uses indexing='ij' (axis 0 = x, axis 1 = y).
xx, yy = np.meshgrid(xs_1d, ys_1d, indexing='ij')   # shape (Nx, Ny)

## 3. SymPy symbols and principal symbol 

In [ ]:
t, x, y   = sp.symbols('t x y', real=True)
xi, eta   = sp.symbols('xi eta', real=True)
u_func    = sp.Function('u')
u_field   = u_func(t, x, y)

# ── Linear symbol in Fourier space ──
# From: ∂u/∂t = ε²·∇²u + ...
# Fourier: ∇² → -(ξ² + η²)
# So: -ε²(ξ² + η²)

k2 = xi**2 + eta**2
symbol_linear = -EPSILON**2 * k2 + 1

print('Principal symbol (linear part):')
print('  a(ξ, η) = ', symbol_linear)

## 4. Allen-Cahn equation 

In [ ]:
# ∂u/∂t = psiOp(-ε²k², u)  +  (u - u³)
#        ───────────────    ─────────
#        Linear diffusion   Double-well reaction

equation = sp.Eq(
    sp.diff(u_field, t),
    psiOp(symbol_linear, u_field) - u_field**3
)

print('Allen-Cahn Equation:')
print('  ∂u/∂t = psiOp(-ε²k², u) + u - u³')

## 5. Initial conditions: Random noise 

In [ ]:
def initial_condition_ac(xx, yy):
    """
    Unstable homogeneous state (u=0) with small random perturbations.
    The nonlinear reaction will quickly amplify the noise, driving the 
    system into the u=+1 and u=-1 phases (spinodal decomposition).
    """
    np.random.seed(99)  # For reproducibility
    noise_amplitude = 0.1
    return noise_amplitude * np.random.randn(xx.shape[0], xx.shape[1])

## 6. Solver setup 

In [ ]:
solver = PDESolver(equation)

solver.setup(
    Lx=Lx, Ly=Ly,
    Nx=Nx, Ny=Ny,
    Lt=Lt, Nt=Nt,
    boundary_condition='periodic',  # Periodic BCs are standard for bulk phase separation
    initial_condition=initial_condition_ac,
    n_frames=n_frames,
    plot=True,
)

## 7. Solve 

In [ ]:
frames = solver.solve()

## 8. Visualization 

In [ ]:
plt.rcParams['animation.embed_limit'] = 2**128

ani = solver.animate(
    component='real',    # Show Re(u) (the physical order parameter)
    overlay=None,  
    mode='imshow',      # 'surface' gives a great 3D view of the sharp interfaces
    physical=True
)

HTML(ani.to_jshtml())

In [ ]:
ani.save('allen_cahn_domain_coarsening.mp4', writer='ffmpeg', fps=20, dpi=100)
print('✅ Saved to allen_cahn_domain_coarsening.mp4')